In [1]:
# DQN
# Q-learning 을 딥러닝 신경망으로 확장한 강화학습 알고리즘
# Q-table 대신 신경망을 사용
# <구조>
'''
환경 Environment
→
현재 상태 state
→
행동별 Q 값 예측
→
행동 action 선택
→
환경에 action 실행
→
reward, next_state, done (경험)
→
ReplayBuffer에 저장
→
랜덤하게 샘플링
→
Target Network 로 target 을 갱신
→
Q-Network 학습 (현재 상태를 입력 받아 각 행동의 Q 값을 갱신)

*** DQN 비유 ***
학생(Q-Network)이 환경에서 문제를 경험함 : Q-Network가 action을 선택해 행동하면
                                            state, action, reward, next_state, done 발생
→ 문제 은행 (ReplayBuffer)에 저장 : state, action, reward, next_state, done
→ 문제 은행에서 랜덤으로 문제를 꺼냄 : Replay Buffer에서 batch sample
→ 정답지(Target Network)를 보고 목표값을 계산함 : target = reward + gamma * max(Target Network(next_state))
→ 학생 신경망을 갱신 : Q-Network 학습, loss = 현재 Q값 - target Q값


~~~ 이전 CartPole 유지하기 예제를 DQN으로 작업 ~~~
[기존 Q-table 방식]         [DQN 방식]
상태를 이산화               연속된 상태 그대로 사용
Q-table로 Q값 예측          신경망 모델이 Q값 예측
argmax(Q[state])            argmax(model.predict(state))
Q[state][action]=r+감마..   model.fit(state, target)으로 현재 상태 갱신
'''

!pip install gymnasium[classic-control]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 42.5 MB/s eta 0:00:00


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
import gymnasium as gym
from collections import deque
import random
import matplotlib.pyplot as plt
import numpy as np

env = gym.make('CartPole-v1')
num_actions = env.action_space.n
state_dim = env.observation_space.shape[0]

In [3]:
def create_model():
    model = Sequential([
        Input(shape=(state_dim,)),
        Dense(64, activation='relu'),
        Dense(64, activation='relu'),
        Dense(int(num_actions), activation='linear')     # 각 행동의 정량적 Q값(제한없는 실수) 출력
    ])
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='mse')
    return model

model = create_model()      # Q-Network (main network) → 매 상태에서 Q(s, a)를 예측하고, model.fit()을 통해 가중치 갱신

# Target Network(학습중에는 고정된 가중치를 유지하며, 일정 주기마다 model의 가중치를 복사함)
# 사용 목적 : Q-learning의 'moving target' 문제를 방지해 학습 안정성 향상
target_model = create_model()
target_model.set_weights(model.get_weights())   # model 과 가중치가 동일하게 초기화

# for i, w in enumerate(model.get_weights()):
#     print(i, '번째 : ', w)

In [4]:
# 하이퍼 파라미터
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995

batch_size = 64     # Replay Buffer 에서 꺼내 학습에 사용하는 샘플 수
# Replay Buffer : 경험(s, a, r, s', done)을 저장해 두는 기억 장소
memory = deque(maxlen=5000)     # Replay Buffer

episodes = 50    # 300 ~ 1000 정도 충분히 줘야 함

# target network 갱신 주기
update_target_every = 5

reward_list = []

In [ ]:
# 학습
for ep in range(episodes):
    state, _ = env.reset()  # 에피소드 시작 시 환경 초기화 - 카트위치, 카트속도, 막대각도, 막대각속도
    done = False
    total_reward = 0

    while not done:
        # 신경망은 batch 형태의 입력을 요구 : [x1, x2, x3, x4] → [[x1, x2, x3, x4]]
        state_input = np.reshape(state, [1, state_dim])

        # 행동 또는 탐험 결정
        if np.random.rand() < epsilon:
            action = np.random.choice(num_actions)
        else:
            q_values = model.predict(state_input, verbose=1)    # 현재 상태에 대한 Q값 예측
            action = np.argmax(q_values)

        # 다음 상태와 보상 받기
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        modified_reward = reward if not done else -10   # 에피소드가 끝난 경우 강하게 벌점 부여

        # 경험을 Replay Buffer에 저장 - 나중에 무작위로 추출해 학습에 참여(샘플의 다양성으로 상관관계 제거)
        memory.append((state, action, modified_reward, next_state, done))
        state = next_state
        total_reward += reward

        # Replay Buffer에 일정 개수 이상 경험이 쌓이면 학습 시작 → Q값 갱신
        if len(memory) >= batch_size:
            minbatch = random.sample(memory, batch_size)
            states, targets = [], []    # 학습을 위해 상태 입력값들과 Q값 target을 저장

            # target 계산 및 갱신 - 밸만 방정식 적용
            for s, a, r, s_next, done in minbatch:
                s_input = np.reshape(s, [1, state_dim])
                s_next_input = np.reshape(s_next, [1, state_dim])

                target = model.predict(s_input, verbose=1)[0]   # [[]] → []
                # 현재 상태에서 모든 행동에 대한 예측된 Q값을 수정한 후 선택한 행동에 대한 Q값만 갱신하고
                # 다시 학습에 사용함
                print('target : ', target)

                if done:
                    target[a] = r   # 종료 상태라면 미래 보상 없음
                else:
                    target[a] = r + gamma * np.max(target_model.predict(s_next_input, verbose=1)[0])
                    # t_next = target_model.predict(s_next_input, verbose=1)[0]

                # 학습용 데이터 쌓기 : 하나(입력 상태, target Q값)의 쌍을 학습 데이터로 저장
                states.append(s)        # 입력 데이터 현재 상태 저장
                targets.append(target)  # 정답 Q 값 저장

            print('\n')
            states = np.array(states)
            targets = np.array(targets)
            print(f'states : {states}, targets : {targets}')
            print('\n')

            model.fit(states, targets, epochs=1, verbose=1)

    reward_list.append(total_reward)

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay
        epsilon = max(epsilon, epsilon_min)

    # Target model 갱신 - target network
    if ep % update_target_every == 0:   # 5번마다 주기적으로 갱신
        target_model.set_weights(model.get_weights())

    if ep % 10 == 0:
        print(f'Episode {ep} : Reward = {total_reward:.1f}, Epsilon = {epsilon:.3f}')

Episode 0 : Reward = 58.0, Epsilon = 0.995
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
target :  [-0.00118903 -0.04029904]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
target :  [0.01684398 0.03887279]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
target :  [0.07039265 0.05543509]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
target :  [0.03145365 0.02861896]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
target :  [-0.01833725 -0.06115585]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
target :  [-0.03668971 -0.07843374]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
target :  [0.01269596 0.05105452]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
target :  [-0.07190053 -0.13418691]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
target :  [0.14154212 0.1176

In [ ]:
plt.plot(reward_list, label="Reward per episode")
plt.plot(moving_average(reward_list, 10), label="Moving average of 10 episodes", color='red')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Training Performance')
plt.show()

# 모델 저장
model.save('cartpole_model.keras')
print('모델 저장 성공')